# Step 3: Energy Forecasting
Train an AI model (Random Forest) to predict energy demand.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

# Load data
energy = pd.read_csv('../data/energy_readings.csv')
weather = pd.read_csv('../data/weather_data.csv')
occupancy = pd.read_csv('../data/occupancy_data.csv')
buildings = pd.read_csv('../data/buildings.csv')

df = energy.merge(weather, on='timestamp').merge(occupancy, on=['timestamp', 'building_id']).merge(buildings, on='building_id')
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Feature Engineering
df['hour'] = df['timestamp'].dt.hour
df['is_weekend'] = df['timestamp'].dt.dayofweek >= 5
df['occ_code'] = df['occupancy_level'].map({'Low':0, 'Medium':1, 'High':2})
df['b_type_code'] = df['building_type'].astype('category').cat.codes

# Train Model
features = ['b_type_code', 'area_sq_m', 'temperature', 'occ_code', 'hour', 'is_weekend']
X = df[features]
y = df['energy_kwh']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)

print(f'MAE: {mean_absolute_error(y_test, model.predict(X_test)):.2f}')
joblib.dump(model, '../models/energy_forecast_model.pkl')

MAE: 0.31


['../models/energy_forecast_model.pkl']